# Datamine KNA Process Example

This notebook demonstrates how to configure and run the **`kna`** process wrapper in `dmstudio`.

## Process Description

## KNA

KNA (Kriging Neighbourhood Analysis) is used to optimize the parameters for grade estimation. Typically multiple estimations are performed on a small number of blocks and the results are averaged for each estimation and compared to find an optimal set of parameters. The parameters are then used by the COKRIG process to estimate grades into a block model by a range of methods including Kriging.

The input to KNA is similar to COKRIG:

* SAMPLES: Samples file
* TESTBLKS: Up to three sets of model blocks to be tested are identified by their XC, YC and ZC coordinates. Each set can represent a different area and is identified by the _BLKGROUP_ field

* EPAR: Specifies estimation parameters such as search volume identifier, kriging method, block size and discretization points

* FIELDS: Input (grade) and output (estimation/**KNA** stats) field names for each estimation

* VMODEL: Contains the variogram model parameters

* SPAR: Contains the search volume parameters referenced in the **EPAR** file

The output file **OUT** lists block size, block group, discretization, search parameters and **KNA** statistics.

### Input Files:

* **samples** (Undefined):
  A file containing sample positional information and supporting attributes.
  Required=Yes

* **testblks** (Undefined):
  A file containing centre positions of test blocks to perform grade estimation on. It
  can optionally contain a numeric field to specify groups of blocks to calculate
  statistics per group which must be specified as the _BLKGROUP_ field.
  Required=Yes

* **epar**:
  The input estimation parameter file used to specify parameters and for each estimation.
  This must contain the following fields:
  Required=No

* **fields** (Undefined):
  A file that contains field names of variables to be used in estimation. Input variables
  must be included under the mandatory column IN_VAR and each of these fields must be
  present in the **SAMPLES** and **VGRAM** file. If more than 1 variable is supplied
  Multivariate (Co)Kriging will be performed. If _IMETHOD_ =10 is used, the column
  _SKMEAN_ must be used to specify the mean per variable. If _IMETHOD_ =11 is used the
  column _LOC_MEAN_ must be used to specify the local mean fields in the prototype model.
  Required=Yes

* **vmodel** (Undefined):
  The input (cross-)variogram model parameter file. If more than 1 variable is suppled in
  the **FIELDS** file (i.e. multivariate estimation), this file must contain the columns
  _GRADE_ and _GRADE2_.
  Required=Yes

* **spar**:
  The input search parameter file. This must contain the following 12 mandatory fields:
  Required=No

### Output Files:

* **out** (Undefined):
  The output file which will contain statistics for each estimation which has been
  performed.
  Required=No

### Fields:

* **key** (Undefined : IN):
  Key field used to specify the field limiting the number of samples for estimation using
  the optional **OPTKEY** and **MAXKEY** parameters in the **SPAR** file. The field must
  exist in the **SAMPLES** file.
  Default=Undefined
  Required=No

* **blkgroup** (Undefined : IN):
  Numeric field in the **PROTO** file used to split test blocks into groups to calculate
  statistics from.
  Default=Undefined
  Required=No

### Parameters:

* **vsetnum**:
  The reference number of the variogram model set to be used from the **VMODEL** file
  Range=Undefined
  Values=Undefined
  Default=1
  Required=Yes

* **nokstats**:
  Set this parameter to 1 to prevent output of Kriging statistics. Typically used with
  BLKCOV=1 when testing the number of discretization points to be used.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **blkcov**:
  Set this parameter to 1 to calculate and output block covariance. Typically used for
  testing discretization.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **nblkcov**:
  The number of random samples to use for calculating block covariance per **KNA** run. A
  value of at least 20 is recommended to ensure reasonable precision.
  Range=Undefined
  Values=Undefined
  Default=20
  Required=No

* **prnt**:
  The level of detail for text printed to the command window. A value of 0 only prints
  errors, a value of 1 additionally prints warnings and progress, a value of 2
  additionally prints further information.
  Range=0,1
  Values=0,1
  Default=0
  Required=Yes

* **nthreads**:
  Number of threads to be used for the main calculation. Any value less than 1 will
  automatically select the values based on the number of virtual cores on the computer.
  Range=Undefined
  Values=Undefined
  Default=-1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('kna')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute kna
print("Running kna...")
dm_cmd.kna(
    samples_i=['t_assays'],  # required
    testblks_i=['t_input_file'],  # required
    fields_i=['t_input_file'],  # required
    vmodel_i='t_mod1',  # required
    spar_i='optional',  # required
    blkgroup_f='optional',  # required
    vsetnum_p='required_val',  # required
    prnt_p='required_val',  # required
    # epar_i='optional',  # optional
    # out_o='t_kna_out',  # optional
    # key_f=['BHID'],  # optional
    # nokstats_p=0,  # optional
    # blkcov_p=0,  # optional
    # nblkcov_p=20,  # optional
    # nthreads_p=-1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("kna execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_kna_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")